# Automated MRI Radiomics Feature Extraction

### TCGA-GBM — Multi-Modal 3D MRI Feature Extraction

This notebook applies the validated MRI radiomics pipeline established in the pilot analysis to the complete available TCGA-GBM imaging dataset.

**Dataset:** TCGA-GBM  
**MRI modalities:** FLAIR, T1, T1Gd, T2  
**Total subjects identified:** 102  
**Subjects successfully processed:** 97  

The pipeline extracts **38 quantitative features per subject**, including tumour geometry, MRI intensity statistics, and 3D GLCM texture features. Features are generated using the manually corrected `GlistrBoost_ManuallyCorrected` tumour segmentation.

The resulting per-subject feature files are combined into a master radiomics table for downstream statistical analysis, machine learning, and integration with molecular and clinical data.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from skimage.measure import marching_cubes
import nibabel as nib
from sklearn.decomposition import PCA
from skimage.feature import graycomatrix

In [2]:
project_dir = Path("..")
data_dir = project_dir / "data"
features_dir = data_dir / "features"

print("Project directory:", project_dir.resolve())
print("Data directory:", data_dir.resolve())
print("Features directory:", features_dir.resolve())

Project directory: C:\Users\Saunak Das\neuro-onco-omics
Data directory: C:\Users\Saunak Das\neuro-onco-omics\data
Features directory: C:\Users\Saunak Das\neuro-onco-omics\data\features


In [3]:
raw_dir = data_dir / "raw"

dataset_dir = raw_dir / "BraTS-TCGA-GBM"

extracted_dir = dataset_dir / "extracted"

subjects_root = (
    extracted_dir
    / "Pre-operative_TCGA_GBM_NIfTI_and_Segmentations"
)

In [4]:
subject_dirs = sorted(
    [
        p for p in subjects_root.iterdir()
        if p.is_dir() and p.name.startswith("TCGA-")
    ]
)

print("Number of subjects:", len(subject_dirs))

print("\nFirst 10 subjects:")
for subject in subject_dirs[:10]:
    print(subject.name)

Number of subjects: 102

First 10 subjects:
TCGA-02-0006
TCGA-02-0009
TCGA-02-0011
TCGA-02-0027
TCGA-02-0033
TCGA-02-0034
TCGA-02-0037
TCGA-02-0046
TCGA-02-0047
TCGA-02-0054


In [5]:
first_subject = subject_dirs[0]

print("Subject:", first_subject.name)
print("\nFiles:")

for item in sorted(first_subject.iterdir()):
    print(item.name)

Subject: TCGA-02-0006

Files:
TCGA-02-0006_1996.08.23_flair.nii.gz
TCGA-02-0006_1996.08.23_GlistrBoost.nii.gz
TCGA-02-0006_1996.08.23_GlistrBoost_ManuallyCorrected.nii.gz
TCGA-02-0006_1996.08.23_t1.nii.gz
TCGA-02-0006_1996.08.23_t1Gd.nii.gz
TCGA-02-0006_1996.08.23_t2.nii.gz


In [6]:
subject = subject_dirs[0]

flair_file = next(subject.glob("*_flair.nii.gz"))
t1_file = next(subject.glob("*_t1.nii.gz"))
t1gd_file = next(subject.glob("*_t1Gd.nii.gz"))
t2_file = next(subject.glob("*_t2.nii.gz"))
mask_file = next(subject.glob("*_GlistrBoost_ManuallyCorrected.nii.gz"))

print("Subject:", subject.name)

print("FLAIR:", flair_file.name)
print("T1:", t1_file.name)
print("T1Gd:", t1gd_file.name)
print("T2:", t2_file.name)
print("Mask:", mask_file.name)

Subject: TCGA-02-0006
FLAIR: TCGA-02-0006_1996.08.23_flair.nii.gz
T1: TCGA-02-0006_1996.08.23_t1.nii.gz
T1Gd: TCGA-02-0006_1996.08.23_t1Gd.nii.gz
T2: TCGA-02-0006_1996.08.23_t2.nii.gz
Mask: TCGA-02-0006_1996.08.23_GlistrBoost_ManuallyCorrected.nii.gz


In [7]:
flair_img = nib.load(flair_file)
t1_img = nib.load(t1_file)
t1gd_img = nib.load(t1gd_file)
t2_img = nib.load(t2_file)
mask_img = nib.load(mask_file)

print("FLAIR shape:", flair_img.shape)
print("T1 shape:", t1_img.shape)
print("T1Gd shape:", t1gd_img.shape)
print("T2 shape:", t2_img.shape)
print("Mask shape:", mask_img.shape)

FLAIR shape: (240, 240, 155)
T1 shape: (240, 240, 155)
T1Gd shape: (240, 240, 155)
T2 shape: (240, 240, 155)
Mask shape: (240, 240, 155)


In [8]:
flair = flair_img.get_fdata()
t1 = t1_img.get_fdata()
t1gd = t1gd_img.get_fdata()
t2 = t2_img.get_fdata()

mask = mask_img.get_fdata()

print("FLAIR:", flair.shape)
print("T1:", t1.shape)
print("T1Gd:", t1gd.shape)
print("T2:", t2.shape)
print("Mask:", mask.shape)

FLAIR: (240, 240, 155)
T1: (240, 240, 155)
T1Gd: (240, 240, 155)
T2: (240, 240, 155)
Mask: (240, 240, 155)


In [9]:
tumour_mask = mask > 0

print("Tumour voxels:", tumour_mask.sum())
print("Mask shape:", tumour_mask.shape)

Tumour voxels: 38314
Mask shape: (240, 240, 155)


In [10]:
images = {
    "FLAIR": flair,
    "T1": t1,
    "T1Gd": t1gd,
    "T2": t2
}

print("Modalities:", list(images.keys()))

Modalities: ['FLAIR', 'T1', 'T1Gd', 'T2']


In [11]:
normalized_images = {}

for modality, image in images.items():

    nonzero_values = image[image > 0]

    p1 = np.percentile(nonzero_values, 1)
    p99 = np.percentile(nonzero_values, 99)

    normalized = (image - p1) / (p99 - p1)

    normalized = np.clip(normalized, 0, 1)

    normalized_images[modality] = normalized

    print(f"{modality}:")
    print(f"  1st percentile: {p1:.2f}")
    print(f"  99th percentile: {p99:.2f}")
    print(f"  Range: {normalized.min():.1f} – {normalized.max():.1f}")

FLAIR:
  1st percentile: 31.00
  99th percentile: 210.00
  Range: 0.0 – 1.0
T1:
  1st percentile: 127.00
  99th percentile: 324.00
  Range: 0.0 – 1.0
T1Gd:
  1st percentile: 195.00
  99th percentile: 438.00
  Range: 0.0 – 1.0
T2:
  1st percentile: 172.00
  99th percentile: 643.00
  Range: 0.0 – 1.0


In [12]:
tumour_images = {}

for modality, image in normalized_images.items():

    tumour_values = image[tumour_mask]

    tumour_images[modality] = tumour_values

    print(f"{modality}:")
    print(f"  Tumour voxels: {len(tumour_values)}")

FLAIR:
  Tumour voxels: 38314
T1:
  Tumour voxels: 38314
T1Gd:
  Tumour voxels: 38314
T2:
  Tumour voxels: 38314


In [13]:
intensity_features = {}

for modality, tumour_values in tumour_images.items():

    mean = np.mean(tumour_values)
    std = np.std(tumour_values)
    cv = std / mean

    intensity_features[modality] = {
        "mean": mean,
        "std": std,
        "cv": cv
    }

    print(f"{modality}:")
    print(f"  Mean: {mean:.3f}")
    print(f"  Std:  {std:.3f}")
    print(f"  CV:   {cv:.3f}")

FLAIR:
  Mean: 0.900
  Std:  0.092
  CV:   0.102
T1:
  Mean: 0.609
  Std:  0.112
  CV:   0.184
T1Gd:
  Mean: 0.569
  Std:  0.101
  CV:   0.178
T2:
  Mean: 0.322
  Std:  0.134
  CV:   0.416


In [14]:
tumour_coords = np.argwhere(tumour_mask)

print("Tumour coordinates shape:", tumour_coords.shape)
print("First 5 coordinates:")
print(tumour_coords[:5])

Tumour coordinates shape: (38314, 3)
First 5 coordinates:
[[131 111  50]
 [131 112  47]
 [131 112  48]
 [131 112  49]
 [131 112  50]]


In [15]:
x_min, y_min, z_min = tumour_coords.min(axis=0)
x_max, y_max, z_max = tumour_coords.max(axis=0)

bbox_x = x_max - x_min + 1
bbox_y = y_max - y_min + 1
bbox_z = z_max - z_min + 1

print("X range:", x_min, "to", x_max)
print("Y range:", y_min, "to", y_max)
print("Z range:", z_min, "to", z_max)

print("\nBounding box:")
print("X:", bbox_x)
print("Y:", bbox_y)
print("Z:", bbox_z)

X range: 131 to 188
Y range: 84 to 142
Z range: 29 to 76

Bounding box:
X: 58
Y: 59
Z: 48


In [16]:
centroid_x = tumour_coords[:, 0].mean()
centroid_y = tumour_coords[:, 1].mean()
centroid_z = tumour_coords[:, 2].mean()

print("Centroid (x, y, z):")
print(
    centroid_x,
    centroid_y,
    centroid_z
)

Centroid (x, y, z):
162.351725217936 110.37876494231874 48.607062692488384


In [17]:
voxel_spacing = mask_img.header.get_zooms()[:3]
print("Voxel spacing (mm):", voxel_spacing)

Voxel spacing (mm): (np.float32(1.0), np.float32(1.0), np.float32(1.0))


In [18]:
voxel_volume = np.prod(voxel_spacing)

tumour_volume_mm3 = tumour_mask.sum() * voxel_volume

print("Volume of one voxel:", voxel_volume, "mm³")
print("Tumour volume:", tumour_volume_mm3, "mm³")

Volume of one voxel: 1.0 mm³
Tumour volume: 38314.0 mm³


In [19]:
verts, faces, normals, values = marching_cubes(
    tumour_mask.astype(np.uint8),
    level=0.5,
    spacing=voxel_spacing
)

surface_area_mm2 = 0.0

for face in faces:
    v1, v2, v3 = verts[face]

    triangle_area = 0.5 * np.linalg.norm(
        np.cross(v2 - v1, v3 - v1)
    )

    surface_area_mm2 += triangle_area

print("Number of surface vertices:", len(verts))
print("Number of surface triangles:", len(faces))
print("Surface area:", surface_area_mm2, "mm²")

Number of surface vertices: 20224
Number of surface triangles: 40924
Surface area: 14430.063 mm²


In [20]:
sphericity = (
    (np.pi ** (1 / 3)) *
    (6 * tumour_volume_mm3) ** (2 / 3)
) / surface_area_mm2

print("Tumour sphericity:", sphericity)

Tumour sphericity: 0.3808794542981205


In [21]:
pca = PCA(n_components=3)
pca.fit(tumour_coords)

spread = np.sqrt(pca.explained_variance_)

elongation = spread[0] / spread[1]

print("Tumour elongation:", elongation)

Tumour elongation: 1.1505380825596834


In [22]:
geometry_features = {
    "tumour_volume_voxels": int(tumour_mask.sum()),
    "tumour_volume_mm3": tumour_volume_mm3,
    "bbox_x": bbox_x,
    "bbox_y": bbox_y,
    "bbox_z": bbox_z,
    "centroid_x": centroid_x,
    "centroid_y": centroid_y,
    "centroid_z": centroid_z,
    "surface_area_mm2": surface_area_mm2,
    "sphericity": sphericity,
    "elongation": elongation
}

print("Geometry features:")
for name, value in geometry_features.items():
    print(f"{name}: {value}")

Geometry features:
tumour_volume_voxels: 38314
tumour_volume_mm3: 38314.0
bbox_x: 58
bbox_y: 59
bbox_z: 48
centroid_x: 162.351725217936
centroid_y: 110.37876494231874
centroid_z: 48.607062692488384
surface_area_mm2: 14430.0634765625
sphericity: 0.3808794542981205
elongation: 1.1505380825596834


In [23]:
quantized_images = {}

for modality, image in normalized_images.items():

    tumour_values = image[tumour_mask]

    quantized_values = np.floor(
        tumour_values * 8
    ).astype(np.uint8)

    quantized_values = np.clip(
        quantized_values,
        0,
        7
    )

    quantized_images[modality] = quantized_values

    print(f"{modality}:")
    print("  Unique tumour levels:", np.unique(quantized_values))

FLAIR:
  Unique tumour levels: [3 4 5 6 7]
T1:
  Unique tumour levels: [0 1 2 3 4 5 6 7]
T1Gd:
  Unique tumour levels: [0 1 2 3 4 5 6 7]
T2:
  Unique tumour levels: [0 1 2 3 4 5 6 7]


In [24]:
quantized_3d_images = {}

for modality, quantized_values in quantized_images.items():

    quantized_3d = np.zeros_like(
        tumour_mask,
        dtype=np.uint8
    )

    quantized_3d[tumour_mask] = quantized_values

    quantized_3d_images[modality] = quantized_3d

    print(f"{modality}:")
    print("  Shape:", quantized_3d.shape)
    print("  Levels:", np.unique(quantized_3d))

FLAIR:
  Shape: (240, 240, 155)
  Levels: [0 3 4 5 6 7]
T1:
  Shape: (240, 240, 155)
  Levels: [0 1 2 3 4 5 6 7]
T1Gd:
  Shape: (240, 240, 155)
  Levels: [0 1 2 3 4 5 6 7]
T2:
  Shape: (240, 240, 155)
  Levels: [0 1 2 3 4 5 6 7]


In [25]:
def calculate_3d_glcm(image, mask, levels=8):

    glcm = np.zeros((levels, levels), dtype=np.float64)

    directions = [
        (1, 0, 0),  
        (0, 1, 0),  
        (0, 0, 1)   
    ]

    for dx, dy, dz in directions:

        source = image[
            0:image.shape[0] - dx,
            0:image.shape[1] - dy,
            0:image.shape[2] - dz
        ]

        neighbour = image[
            dx:image.shape[0],
            dy:image.shape[1],
            dz:image.shape[2]
        ]

        source_mask = mask[
            0:mask.shape[0] - dx,
            0:mask.shape[1] - dy,
            0:mask.shape[2] - dz
        ]

        neighbour_mask = mask[
            dx:mask.shape[0],
            dy:mask.shape[1],
            dz:mask.shape[2]
        ]

        valid = source_mask & neighbour_mask

        i = source[valid]
        j = neighbour[valid]

        for a, b in zip(i, j):
            glcm[a, b] += 1

    glcm = glcm + glcm.T

    if glcm.sum() > 0:
        glcm = glcm / glcm.sum()

    return glcm

In [26]:
flair_quantized_3d = quantized_3d_images["FLAIR"]

glcm_3d = calculate_3d_glcm(
    flair_quantized_3d,
    tumour_mask,
    levels=8
)

print("3D GLCM shape:", glcm_3d.shape)
print("GLCM sum:", glcm_3d.sum())

3D GLCM shape: (8, 8)
GLCM sum: 1.0


In [27]:
i, j = np.meshgrid(
    np.arange(8),
    np.arange(8),
    indexing="ij"
)

contrast_3d = np.sum(
    glcm_3d * (i - j) ** 2
)

homogeneity_3d = np.sum(
    glcm_3d / (1 + np.abs(i - j))
)

energy_3d = np.sqrt(
    np.sum(glcm_3d ** 2)
)

row = np.arange(8)
col = np.arange(8)

mu_i = np.sum(row * glcm_3d.sum(axis=1))
mu_j = np.sum(col * glcm_3d.sum(axis=0))

sigma_i = np.sqrt(
    np.sum((row - mu_i) ** 2 * glcm_3d.sum(axis=1))
)

sigma_j = np.sqrt(
    np.sum((col - mu_j) ** 2 * glcm_3d.sum(axis=0))
)

correlation_3d = np.sum(
    (row[:, None] - mu_i) *
    (col[None, :] - mu_j) *
    glcm_3d
) / (sigma_i * sigma_j)


print("FLAIR 3D GLCM contrast:", contrast_3d)
print("FLAIR 3D GLCM homogeneity:", homogeneity_3d)
print("FLAIR 3D GLCM energy:", energy_3d)
print("FLAIR 3D GLCM correlation:", correlation_3d)

FLAIR 3D GLCM contrast: 0.17873723748057618
FLAIR 3D GLCM homogeneity: 0.9147862506395162
FLAIR 3D GLCM energy: 0.6172493984106854
FLAIR 3D GLCM correlation: 0.735401506080346


In [28]:
def calculate_glcm_features(glcm):

    levels = glcm.shape[0]

    i, j = np.meshgrid(
        np.arange(levels),
        np.arange(levels),
        indexing="ij"
    )

    contrast = np.sum(
        glcm * (i - j) ** 2
    )

    homogeneity = np.sum(
        glcm / (1 + np.abs(i - j))
    )

    energy = np.sqrt(
        np.sum(glcm ** 2)
    )

    p_i = glcm.sum(axis=1)
    p_j = glcm.sum(axis=0)

    mu_i = np.sum(
        np.arange(levels) * p_i
    )

    mu_j = np.sum(
        np.arange(levels) * p_j
    )

    sigma_i = np.sqrt(
        np.sum(
            (np.arange(levels) - mu_i) ** 2 * p_i
        )
    )

    sigma_j = np.sqrt(
        np.sum(
            (np.arange(levels) - mu_j) ** 2 * p_j
        )
    )

    if sigma_i > 0 and sigma_j > 0:
        correlation = np.sum(
            (i - mu_i) *
            (j - mu_j) *
            glcm
        ) / (sigma_i * sigma_j)
    else:
        correlation = 0.0

    return {
        "contrast": contrast,
        "homogeneity": homogeneity,
        "energy": energy,
        "correlation": correlation
    }

In [29]:
glcm_features = {}

for modality in ["FLAIR", "T1", "T1Gd", "T2"]:

    glcm = calculate_3d_glcm(
        quantized_3d_images[modality],
        tumour_mask,
        levels=8
    )

    features = calculate_glcm_features(glcm)

    glcm_features[modality] = features

    print(f"{modality}:")
    print(f"  contrast: {features['contrast']:.4f}")
    print(f"  homogeneity: {features['homogeneity']:.4f}")
    print(f"  energy: {features['energy']:.4f}")
    print(f"  correlation: {features['correlation']:.4f}")

FLAIR:
  contrast: 0.1787
  homogeneity: 0.9148
  energy: 0.6172
  correlation: 0.7354
T1:
  contrast: 0.2445
  homogeneity: 0.8815
  energy: 0.4305
  correlation: 0.8595
T1Gd:
  contrast: 0.2729
  homogeneity: 0.8751
  energy: 0.4657
  correlation: 0.8086
T2:
  contrast: 0.2412
  homogeneity: 0.8875
  energy: 0.4496
  correlation: 0.9030


In [31]:
intensity_features = {}

for modality, values in tumour_images.items():
    mean = np.mean(values)
    std = np.std(values)
    cv = std / mean if mean != 0 else 0

    intensity_features[f"{modality}_mean"] = mean
    intensity_features[f"{modality}_std"] = std
    intensity_features[f"{modality}_cv"] = cv

for name, value in intensity_features.items():
    print(f"{name}: {value}")

FLAIR_mean: 0.8999359307667341
FLAIR_std: 0.09192837518845405
FLAIR_cv: 0.10214991094991867
T1_mean: 0.608783843045272
T1_std: 0.11180433775507523
T1_cv: 0.1836519464705323
T1Gd_mean: 0.5687273087382128
T1Gd_std: 0.10121077578175842
T1Gd_cv: 0.17796011238902212
T2_mean: 0.3223997658414706
T2_std: 0.13412939860999556
T2_cv: 0.41603441696030646


In [32]:
all_features = {
    "tumour_volume_voxels": geometry_features["tumour_volume_voxels"],
    "bbox_x": geometry_features["bbox_x"],
    "bbox_y": geometry_features["bbox_y"],
    "bbox_z": geometry_features["bbox_z"],
    "centroid_x": geometry_features["centroid_x"],
    "centroid_y": geometry_features["centroid_y"],
    "centroid_z": geometry_features["centroid_z"],
    "surface_area_mm2": geometry_features["surface_area_mm2"],
    "sphericity": geometry_features["sphericity"],
    "elongation": geometry_features["elongation"]
}

all_features.update(intensity_features)

for modality in ["FLAIR", "T1", "T1Gd", "T2"]:
    for feature_name in ["contrast", "homogeneity", "energy", "correlation"]:
        all_features[f"{modality}_glcm_{feature_name}"] = glcm_features[modality][feature_name]

features_df = pd.DataFrame([all_features])

print("Shape:", features_df.shape)
display(features_df)

Shape: (1, 38)


,tumour_volume_voxels,bbox_x,bbox_y,bbox_z,centroid_x,centroid_y,centroid_z,surface_area_mm2,sphericity,elongation,...,T1_glcm_energy,T1_glcm_correlation,T1Gd_glcm_contrast,T1Gd_glcm_homogeneity,T1Gd_glcm_energy,T1Gd_glcm_correlation,T2_glcm_contrast,T2_glcm_homogeneity,T2_glcm_energy,T2_glcm_correlation
0,38314,58,59,48,162.351725,110.378765,48.607063,14430.063477,0.380879,1.150538,...,0.430453,0.859521,0.272925,0.875081,0.465681,0.808577,0.241189,0.887527,0.449627,0.903


In [35]:
per_subject_dir = features_dir / "per_subject"
per_subject_dir.mkdir(parents=True, exist_ok=True)

print("Created:", per_subject_dir)

Created: ..\data\features\per_subject


In [36]:
output_path = per_subject_dir / f"{subject.name}_features.csv"
features_df.to_csv(output_path, index=False)
print("Saved to:", output_path)

Saved to: ..\data\features\per_subject\TCGA-02-0006_features.csv


In [37]:
def extract_subject_features(subject):

    flair_file = next(subject.glob("*_flair.nii.gz"))
    t1_file = next(subject.glob("*_t1.nii.gz"))
    t1gd_file = next(subject.glob("*_t1Gd.nii.gz"))
    t2_file = next(subject.glob("*_t2.nii.gz"))
    mask_file = next(subject.glob("*_GlistrBoost_ManuallyCorrected.nii.gz"))

    flair_img = nib.load(flair_file)
    t1_img = nib.load(t1_file)
    t1gd_img = nib.load(t1gd_file)
    t2_img = nib.load(t2_file)
    mask_img = nib.load(mask_file)

    images = {
        "FLAIR": flair_img.get_fdata(),
        "T1": t1_img.get_fdata(),
        "T1Gd": t1gd_img.get_fdata(),
        "T2": t2_img.get_fdata()
    }

    mask = mask_img.get_fdata()
    tumour_mask = mask > 0

    normalized_images = {}

    for modality, image in images.items():
        nonzero_values = image[image > 0]

        p1 = np.percentile(nonzero_values, 1)
        p99 = np.percentile(nonzero_values, 99)

        normalized = (image - p1) / (p99 - p1)
        normalized = np.clip(normalized, 0, 1)

        normalized_images[modality] = normalized

    tumour_images = {}

    for modality, image in normalized_images.items():
        tumour_images[modality] = image[tumour_mask]

    tumour_coords = np.argwhere(tumour_mask)

    x_min, y_min, z_min = tumour_coords.min(axis=0)
    x_max, y_max, z_max = tumour_coords.max(axis=0)

    bbox_x = x_max - x_min + 1
    bbox_y = y_max - y_min + 1
    bbox_z = z_max - z_min + 1

    centroid_x = tumour_coords[:, 0].mean()
    centroid_y = tumour_coords[:, 1].mean()
    centroid_z = tumour_coords[:, 2].mean()

    voxel_spacing = mask_img.header.get_zooms()[:3]
    voxel_volume = np.prod(voxel_spacing)

    tumour_volume_voxels = int(tumour_mask.sum())
    tumour_volume_mm3 = tumour_volume_voxels * voxel_volume

    verts, faces, normals, values = marching_cubes(
        tumour_mask.astype(np.uint8),
        level=0.5,
        spacing=voxel_spacing
    )

    surface_area_mm2 = 0.0

    for face in faces:
        v1, v2, v3 = verts[face]
        triangle_area = 0.5 * np.linalg.norm(
            np.cross(v2 - v1, v3 - v1)
        )
        surface_area_mm2 += triangle_area

    sphericity = (
        (np.pi ** (1 / 3)) *
        (6 * tumour_volume_mm3) ** (2 / 3)
    ) / surface_area_mm2

    pca = PCA(n_components=3)
    pca.fit(tumour_coords)

    spread = np.sqrt(pca.explained_variance_)
    elongation = spread[0] / spread[1]

    geometry_features = {
        "tumour_volume_voxels": tumour_volume_voxels,
        "bbox_x": bbox_x,
        "bbox_y": bbox_y,
        "bbox_z": bbox_z,
        "centroid_x": centroid_x,
        "centroid_y": centroid_y,
        "centroid_z": centroid_z,
        "surface_area_mm2": surface_area_mm2,
        "sphericity": sphericity,
        "elongation": elongation
    }

    intensity_features = {}

    for modality, values in tumour_images.items():
        mean = np.mean(values)
        std = np.std(values)
        cv = std / mean if mean != 0 else 0

        intensity_features[f"{modality}_mean"] = mean
        intensity_features[f"{modality}_std"] = std
        intensity_features[f"{modality}_cv"] = cv

    quantized_images = {}

    for modality, image in normalized_images.items():
        tumour_values = image[tumour_mask]

        quantized_values = np.floor(
            tumour_values * 8
        ).astype(np.uint8)

        quantized_values = np.clip(
            quantized_values,
            0,
            7
        )

        quantized_images[modality] = quantized_values

    quantized_3d_images = {}

    for modality, quantized_values in quantized_images.items():
        quantized_3d = np.zeros_like(
            tumour_mask,
            dtype=np.uint8
        )

        quantized_3d[tumour_mask] = quantized_values

        quantized_3d_images[modality] = quantized_3d

    glcm_features = {}

    for modality in ["FLAIR", "T1", "T1Gd", "T2"]:

        glcm = calculate_3d_glcm(
            quantized_3d_images[modality],
            tumour_mask,
            levels=8
        )

        glcm_features[modality] = calculate_glcm_features(glcm)

    all_features = geometry_features.copy()
    all_features.update(intensity_features)

    for modality in ["FLAIR", "T1", "T1Gd", "T2"]:
        for feature_name in [
            "contrast",
            "homogeneity",
            "energy",
            "correlation"
        ]:
            all_features[
                f"{modality}_glcm_{feature_name}"
            ] = glcm_features[modality][feature_name]

    return pd.DataFrame([all_features])

In [38]:
test_features = extract_subject_features(subject)

print("Shape:", test_features.shape)
display(test_features)

Shape: (1, 38)


,tumour_volume_voxels,bbox_x,bbox_y,bbox_z,centroid_x,centroid_y,centroid_z,surface_area_mm2,sphericity,elongation,...,T1_glcm_energy,T1_glcm_correlation,T1Gd_glcm_contrast,T1Gd_glcm_homogeneity,T1Gd_glcm_energy,T1Gd_glcm_correlation,T2_glcm_contrast,T2_glcm_homogeneity,T2_glcm_energy,T2_glcm_correlation
0,38314,58,59,48,162.351725,110.378765,48.607063,14430.063477,0.38088,1.150538,...,0.430453,0.859521,0.272925,0.875081,0.465681,0.808577,0.241189,0.887527,0.449627,0.903


In [39]:
output_path = per_subject_dir / f"{subject.name}_features.csv"
test_features.to_csv(output_path, index=False)
print("Saved to:", output_path)

Saved to: ..\data\features\per_subject\TCGA-02-0006_features.csv


In [40]:
expected_columns = features_df.columns.tolist()

completed = []
failed = []

for i, subject in enumerate(subject_dirs, 1):

    try:
        subject_features = extract_subject_features(subject)

        if subject_features.shape != (1, 38):
            raise ValueError(
                f"Expected 38 features, got {subject_features.shape[1]}"
            )

        subject_features = subject_features[expected_columns]

        output_path = per_subject_dir / f"{subject.name}_features.csv"
        subject_features.to_csv(output_path, index=False)

        completed.append(subject.name)

        print(f"[{i}/{len(subject_dirs)}] {subject.name} ✓")

    except Exception as e:
        failed.append((subject.name, str(e)))

        print(f"[{i}/{len(subject_dirs)}] {subject.name} ✗")
        print(f"    {e}")

print("\nCompleted:", len(completed))
print("Failed:", len(failed))

[1/102] TCGA-02-0006 ✓
[2/102] TCGA-02-0009 ✓
[3/102] TCGA-02-0011 ✓
[4/102] TCGA-02-0027 ✓
[5/102] TCGA-02-0033 ✓
[6/102] TCGA-02-0034 ✓
[7/102] TCGA-02-0037 ✓
[8/102] TCGA-02-0046 ✓
[9/102] TCGA-02-0047 ✓
[10/102] TCGA-02-0054 ✓
[11/102] TCGA-02-0059 ✓
[12/102] TCGA-02-0064 ✓
[13/102] TCGA-02-0068 ✓
[14/102] TCGA-02-0069 ✓
[15/102] TCGA-02-0070 ✗
    
[16/102] TCGA-02-0075 ✓
[17/102] TCGA-02-0085 ✓
[18/102] TCGA-02-0086 ✓
[19/102] TCGA-02-0087 ✓
[20/102] TCGA-02-0102 ✓
[21/102] TCGA-02-0106 ✓
[22/102] TCGA-02-0116 ✓
[23/102] TCGA-06-0119 ✓
[24/102] TCGA-06-0122 ✓
[25/102] TCGA-06-0130 ✓
[26/102] TCGA-06-0137 ✓
[27/102] TCGA-06-0138 ✓
[28/102] TCGA-06-0139 ✓
[29/102] TCGA-06-0142 ✓
[30/102] TCGA-06-0145 ✓
[31/102] TCGA-06-0149 ✓
[32/102] TCGA-06-0154 ✓
[33/102] TCGA-06-0158 ✓
[34/102] TCGA-06-0162 ✓
[35/102] TCGA-06-0164 ✓
[36/102] TCGA-06-0176 ✓
[37/102] TCGA-06-0177 ✓
[38/102] TCGA-06-0179 ✓
[39/102] TCGA-06-0182 ✓
[40/102] TCGA-06-0184 ✓
[41/102] TCGA-06-0185 ✓
[42/102] TCGA-06-018

In [41]:
for subject_name, error in failed:
    print(subject_name)
    print(error)
    print()

TCGA-02-0070


TCGA-06-0238


TCGA-08-0509


TCGA-08-0520


TCGA-12-3650




In [42]:
failed_subjects = [
    "TCGA-02-0070",
    "TCGA-06-0238",
    "TCGA-08-0509",
    "TCGA-08-0520",
    "TCGA-12-3650"
]

for subject_name in failed_subjects:
    subject = subjects_root / subject_name

    print(f"\n{subject_name}")
    for file in sorted(subject.glob("*.nii.gz")):
        print(" ", file.name)


TCGA-02-0070
  TCGA-02-0070_2000.07.10_flair.nii.gz
  TCGA-02-0070_2000.07.10_GlistrBoost.nii.gz
  TCGA-02-0070_2000.07.10_t1.nii.gz
  TCGA-02-0070_2000.07.10_t1Gd.nii.gz
  TCGA-02-0070_2000.07.10_t2.nii.gz

TCGA-06-0238
  TCGA-06-0238_2005.04.12_flair.nii.gz
  TCGA-06-0238_2005.04.12_GlistrBoost.nii.gz
  TCGA-06-0238_2005.04.12_t1.nii.gz
  TCGA-06-0238_2005.04.12_t1Gd.nii.gz
  TCGA-06-0238_2005.04.12_t2.nii.gz

TCGA-08-0509
  TCGA-08-0509_1997.11.13_flair.nii.gz
  TCGA-08-0509_1997.11.13_GlistrBoost.nii.gz
  TCGA-08-0509_1997.11.13_t1.nii.gz
  TCGA-08-0509_1997.11.13_t1Gd.nii.gz
  TCGA-08-0509_1997.11.13_t2.nii.gz

TCGA-08-0520
  TCGA-08-0520_1999.10.13_flair.nii.gz
  TCGA-08-0520_1999.10.13_GlistrBoost.nii.gz
  TCGA-08-0520_1999.10.13_t1.nii.gz
  TCGA-08-0520_1999.10.13_t1Gd.nii.gz
  TCGA-08-0520_1999.10.13_t2.nii.gz

TCGA-12-3650
  TCGA-12-3650_2001.07.29_flair.nii.gz
  TCGA-12-3650_2001.07.29_GlistrBoost.nii.gz
  TCGA-12-3650_2001.07.29_t1.nii.gz
  TCGA-12-3650_2001.07.29_t1Gd.nii

In [43]:
feature_files = sorted(per_subject_dir.glob("*_features.csv"))

print("Feature files found:", len(feature_files))

for file in feature_files[:10]:
    print(file.name)

Feature files found: 97
TCGA-02-0006_features.csv
TCGA-02-0009_features.csv
TCGA-02-0011_features.csv
TCGA-02-0027_features.csv
TCGA-02-0033_features.csv
TCGA-02-0034_features.csv
TCGA-02-0037_features.csv
TCGA-02-0046_features.csv
TCGA-02-0047_features.csv
TCGA-02-0054_features.csv


In [44]:
master_features = []

for file in feature_files:
    subject_id = file.stem.replace("_features", "")
    
    df = pd.read_csv(file)
    df.insert(0, "subject_id", subject_id)
    
    master_features.append(df)

master_df = pd.concat(master_features, ignore_index=True)

print("Shape:", master_df.shape)
display(master_df.head())

Shape: (97, 39)


,subject_id,tumour_volume_voxels,bbox_x,bbox_y,bbox_z,centroid_x,centroid_y,centroid_z,surface_area_mm2,sphericity,...,T1_glcm_energy,T1_glcm_correlation,T1Gd_glcm_contrast,T1Gd_glcm_homogeneity,T1Gd_glcm_energy,T1Gd_glcm_correlation,T2_glcm_contrast,T2_glcm_homogeneity,T2_glcm_energy,T2_glcm_correlation
0,TCGA-02-0006,38314,58,59,48,162.351725,110.378765,48.607063,14430.063,0.380880,...,0.430453,0.859521,0.272925,0.875081,0.465681,0.808577,0.241189,0.887527,0.449627,0.903000
1,TCGA-02-0009,24434,44,68,45,81.841982,139.207457,89.537530,8650.995,0.470704,...,0.431452,0.884143,0.264455,0.878143,0.411409,0.872941,0.170629,0.917317,0.585403,0.781443
2,TCGA-02-0011,127812,58,88,86,129.358902,135.633110,83.403796,23416.736,0.524009,...,0.372208,0.921169,0.339214,0.860104,0.378785,0.879776,0.275246,0.878699,0.329869,0.949014
3,TCGA-02-0027,53787,52,65,52,99.849257,100.725528,92.431089,11517.636,0.598282,...,0.462694,0.902516,0.501639,0.821439,0.316014,0.872944,0.353855,0.852461,0.339439,0.909701
4,TCGA-02-0033,107328,73,95,81,103.393010,155.991717,85.899346,19205.742,0.568672,...,0.452580,0.848699,0.320419,0.877294,0.409398,0.927798,0.292300,0.865855,0.382581,0.894469


In [45]:
master_output_path = features_dir / "tcga_gbm_radiomics_features.csv"
master_df.to_csv(master_output_path, index=False)
print("Saved to:", master_output_path)

Saved to: ..\data\features\tcga_gbm_radiomics_features.csv


In [48]:
print("Rows:", len(master_df))
print("Columns:", len(master_df.columns))
print("Duplicate subjects:", master_df["subject_id"].duplicated().sum())
print("Missing values:", master_df.isna().sum().sum())
print("\nUnique subjects:", master_df["subject_id"].nunique())

Rows: 97
Columns: 39
Duplicate subjects: 0
Missing values: 0

Unique subjects: 97


In [50]:
pilot_features = pd.read_csv(
    r"C:\Users\Saunak Das\neuro-onco-omics\data\features\patient_001_features.csv"
)

pilot_columns = pilot_features.columns.tolist()
master_columns = master_df.drop(columns="subject_id").columns.tolist()

print("Same columns:", pilot_columns == master_columns)

print("\nMissing from master:")
print(set(pilot_columns) - set(master_columns))

print("\nExtra in master:")
print(set(master_columns) - set(pilot_columns))

Same columns: True

Missing from master:
set()

Extra in master:
set()


In [51]:
failed_log = pd.DataFrame({
    "subject_id": failed_subjects,
    "status": "excluded",
    "reason": "GlistrBoost_ManuallyCorrected segmentation unavailable"
})

failed_log_path = features_dir / "failed_subjects.csv"

failed_log.to_csv(failed_log_path, index=False)

print("Saved to:", failed_log_path)
display(failed_log)

Saved to: ..\data\features\failed_subjects.csv


,subject_id,status,reason
0,TCGA-02-0070,excluded,GlistrBoost_ManuallyCorrected segmentation una...
1,TCGA-06-0238,excluded,GlistrBoost_ManuallyCorrected segmentation una...
2,TCGA-08-0509,excluded,GlistrBoost_ManuallyCorrected segmentation una...
3,TCGA-08-0520,excluded,GlistrBoost_ManuallyCorrected segmentation una...
4,TCGA-12-3650,excluded,GlistrBoost_ManuallyCorrected segmentation una...


In [52]:
feature_data = master_df.drop(columns="subject_id")
print("Non-numeric values:", feature_data.select_dtypes(exclude="number").shape[1])
print("NaN values:", feature_data.isna().sum().sum())
print("Infinite values:", np.isinf(feature_data.to_numpy()).sum())
print("Duplicate columns:", feature_data.columns.duplicated().sum())

Non-numeric values: 0
NaN values: 0
Infinite values: 0
Duplicate columns: 0


In [53]:
feature_summary = master_df.drop(columns="subject_id").describe().T
display(feature_summary)

,count,mean,std,min,25%,50%,75%,max
tumour_volume_voxels,97.0,109875.371134,51907.134841,12952.000000,67704.000000,105852.000000,156606.000000,227126.000000
bbox_x,97.0,63.134021,12.953144,33.000000,52.000000,61.000000,73.000000,97.000000
bbox_y,97.0,85.195876,18.307352,39.000000,73.000000,85.000000,97.000000,129.000000
bbox_z,97.0,68.144330,13.589204,32.000000,58.000000,70.000000,78.000000,95.000000
centroid_x,97.0,122.413279,31.138931,69.890485,90.880833,135.737284,152.730465,164.096197
centroid_y,97.0,132.543672,23.787116,81.758559,114.441095,129.869812,151.115168,181.101164
centroid_z,97.0,76.591503,15.580245,47.174219,64.692777,73.383040,90.507529,107.561250
surface_area_mm2,97.0,20891.175104,8320.057147,5361.054700,14267.895000,19508.060000,27253.390000,41654.360000
sphericity,97.0,0.538690,0.106635,0.230415,0.465284,0.544660,0.631890,0.739097
elongation,97.0,1.349892,0.193634,1.006207,1.217340,1.309049,1.442612,2.056619


In [55]:
print("Sphericity > 1:", (master_df["sphericity"] > 1).sum())
print("Sphericity <= 0:", (master_df["sphericity"] <= 0).sum())

print("Elongation < 1:", (master_df["elongation"] < 1).sum())

for modality in ["FLAIR", "T1", "T1Gd", "T2"]:
    print(f"\n{modality}")
    print("Mean outside 0–1:", ((master_df[f"{modality}_mean"] < 0) | (master_df[f"{modality}_mean"] > 1)).sum())
    print("Std < 0:", (master_df[f"{modality}_std"] < 0).sum())
    print("CV < 0:", (master_df[f"{modality}_cv"] < 0).sum())
    print("GLCM contrast < 0:", (master_df[f"{modality}_glcm_contrast"] < 0).sum())
    print("GLCM homogeneity outside 0–1:",
          ((master_df[f"{modality}_glcm_homogeneity"] < 0) |
           (master_df[f"{modality}_glcm_homogeneity"] > 1)).sum())
    print("GLCM energy outside 0–1:",
          ((master_df[f"{modality}_glcm_energy"] < 0) |
           (master_df[f"{modality}_glcm_energy"] > 1)).sum())
    print("GLCM correlation outside -1–1:",
          ((master_df[f"{modality}_glcm_correlation"] < -1) |
           (master_df[f"{modality}_glcm_correlation"] > 1)).sum())

Sphericity > 1: 0
Sphericity <= 0: 0
Elongation < 1: 0

FLAIR
Mean outside 0–1: 0
Std < 0: 0
CV < 0: 0
GLCM contrast < 0: 0
GLCM homogeneity outside 0–1: 0
GLCM energy outside 0–1: 0
GLCM correlation outside -1–1: 0

T1
Mean outside 0–1: 0
Std < 0: 0
CV < 0: 0
GLCM contrast < 0: 0
GLCM homogeneity outside 0–1: 0
GLCM energy outside 0–1: 0
GLCM correlation outside -1–1: 0

T1Gd
Mean outside 0–1: 0
Std < 0: 0
CV < 0: 0
GLCM contrast < 0: 0
GLCM homogeneity outside 0–1: 0
GLCM energy outside 0–1: 0
GLCM correlation outside -1–1: 0

T2
Mean outside 0–1: 0
Std < 0: 0
CV < 0: 0
GLCM contrast < 0: 0
GLCM homogeneity outside 0–1: 0
GLCM energy outside 0–1: 0
GLCM correlation outside -1–1: 0


In [56]:
master_output_path = features_dir / "tcga_gbm_radiomics_features.csv"

master_df.to_csv(master_output_path, index=False)

print("Final master table saved to:")
print(master_output_path)

Final master table saved to:
..\data\features\tcga_gbm_radiomics_features.csv


# Conclusion and Detailed Workflow Summary

## 1. Objective

This notebook extends the validated pilot MRI feature-extraction workflow from Notebook 01 `01_piolot_analysis.ipynb` to the complete available TCGA-GBM imaging dataset.

The purpose is to generate a reproducible subject-level radiomics dataset from pre-operative multi-modal brain MRI and tumour segmentations. The extracted imaging features are intended to provide the quantitative imaging foundation for downstream statistical analysis, machine-learning modelling, and integration with molecular and clinical information.

The pipeline was first established and checked on a pilot subject and was then applied systematically to the full set of available subjects.

---

## 2. Dataset

The dataset used in this notebook contains pre-operative MRI data from TCGA-GBM subjects.

The imaging modalities processed were:

- **FLAIR**
- **T1**
- **T1Gd**
- **T2**

For tumour delineation, the pipeline uses the:

`*_GlistrBoost_ManuallyCorrected.nii.gz`

segmentation whenever it is available.

A total of **102 TCGA-GBM subject directories** were identified in the extracted dataset.

---

## 3. Input Data Structure

The extracted dataset follows a subject-based structure in which each TCGA subject has a separate directory containing the MRI modalities and tumour segmentation.

Representative files include:

- `*_flair.nii.gz`
- `*_t1.nii.gz`
- `*_t1Gd.nii.gz`
- `*_t2.nii.gz`
- `*_GlistrBoost_ManuallyCorrected.nii.gz`

The subject identifier is taken directly from the original TCGA directory name.

Examples:

`TCGA-02-0006`

`TCGA-02-0009`

`TCGA-02-0011`

Using the original TCGA identifiers avoids introducing artificial patient numbering and preserves the link between the extracted features and the source dataset.

---

## 4. MRI Loading

The NIfTI MRI volumes were loaded using `nibabel`.

For each subject, the four MRI modalities and the tumour segmentation were loaded independently.

The voxel arrays were extracted using:

`get_fdata()`

The segmentation was converted into a binary tumour mask:

`tumour_mask = mask > 0`

The mask therefore represents all voxels labelled as tumour by the selected segmentation.

---

## 5. Intensity Normalization

MRI signal intensities are not directly comparable across scans because their numerical intensity scales are not standardized.

To reduce the influence of extreme intensity values while retaining the relative intensity structure of each scan, normalization was performed separately for each MRI modality.

Only non-zero image voxels were used to calculate the intensity percentiles so that the large background region did not dominate the percentile calculation.

For each modality:

1. All non-zero image intensities were extracted.
2. The **1st percentile** was calculated.
3. The **99th percentile** was calculated.
4. Intensities were linearly scaled between these two values.
5. Values were clipped to the range **0–1**.

The normalization can be represented as:

`normalized = (image - p1) / (p99 - p1)`

followed by:

`normalized = clip(normalized, 0, 1)`

This normalization was applied independently to:

- FLAIR
- T1
- T1Gd
- T2

For the pilot subject, the normalization parameters reproduced the values established during the pilot analysis:

- FLAIR: 1st percentile = 31, 99th percentile = 210
- T1: 1st percentile = 127, 99th percentile = 324
- T1Gd: 1st percentile = 195, 99th percentile = 438
- T2: 1st percentile = 172, 99th percentile = 643

This confirmed consistency between the pilot and the automated pipeline.

---

## 6. Tumour-Restricted Feature Extraction

After normalization, image intensities were restricted to the tumour mask.

For each modality, only voxels satisfying:

`tumour_mask == True`

were used for intensity and texture feature extraction.

This ensures that the calculated radiomics features describe the tumour region rather than the entire brain volume or background.

Tumour voxel coordinates were additionally extracted from the binary mask and used for the geometric calculations.

---

# 7. Radiomics Feature Set

A total of **38 features** were extracted for every successfully processed subject.

The feature set consists of:

- **10 tumour geometry features**
- **12 intensity features**
- **16 3D GLCM texture features**

Therefore:

`10 + 12 + 16 = 38 features per subject`

---

## 8. Tumour Geometry Features

Ten geometric features were calculated from the tumour mask.

### 8.1 Tumour Volume

The total number of tumour voxels was calculated:

`tumour_volume_voxels`

This represents the tumour size in voxel units.

### 8.2 Bounding Box

The minimum and maximum tumour coordinates along each spatial axis were determined.

The resulting axis-aligned bounding-box dimensions are:

- `bbox_x`
- `bbox_y`
- `bbox_z`

The `+1` correction is applied when converting coordinate ranges into voxel dimensions.

### 8.3 Tumour Centroid

The mean coordinate of all tumour voxels was calculated to obtain the tumour centroid:

- `centroid_x`
- `centroid_y`
- `centroid_z`

These coordinates describe the centre of the tumour mask in voxel space.

### 8.4 Surface Area

The tumour surface was reconstructed from the binary segmentation using the marching-cubes algorithm.

The voxel spacing obtained from the NIfTI header was supplied to the surface reconstruction so that the resulting surface area is expressed in physical units.

The resulting feature is:

`surface_area_mm2`

and is reported in square millimetres.

### 8.5 Sphericity

Tumour sphericity quantifies how closely the tumour shape approaches a sphere.

It is calculated from tumour volume and surface area using:

`Sphericity = π^(1/3) × (6V)^(2/3) / A`

where:

- `V` = tumour volume
- `A` = tumour surface area

A value closer to 1 represents a shape closer to a sphere, whereas lower values indicate greater deviation from a spherical shape.

### 8.6 Elongation

Tumour shape anisotropy was estimated using principal component analysis (PCA) on the 3D tumour coordinates.

The spatial spread along the principal axes was obtained from the PCA explained variance.

Elongation was calculated as:

`largest principal spread / second-largest principal spread`

The resulting feature is:

`elongation`

Values greater than 1 indicate that the tumour extends more along its first principal axis than along its second principal axis.

---

# 9. MRI Intensity Features

Three intensity features were calculated for each of the four MRI modalities.

The tumour-restricted normalized intensities were used.

### FLAIR

- `FLAIR_mean`
- `FLAIR_std`
- `FLAIR_cv`

### T1

- `T1_mean`
- `T1_std`
- `T1_cv`

### T1Gd

- `T1Gd_mean`
- `T1Gd_std`
- `T1Gd_cv`

### T2

- `T2_mean`
- `T2_std`
- `T2_cv`

This gives:

`4 modalities × 3 intensity features = 12 intensity features`

### 9.1 Mean

The mean tumour intensity represents the average normalized MRI signal within the tumour.

### 9.2 Standard Deviation

The standard deviation quantifies the variability of normalized intensity values within the tumour.

### 9.3 Coefficient of Variation

The coefficient of variation was calculated as:

`CV = standard deviation / mean`

It provides a measure of relative intensity heterogeneity that accounts for differences in mean intensity.

---

# 10. 3D GLCM Texture Features

Texture features were calculated from a custom **3D Gray-Level Co-occurrence Matrix (GLCM)** implementation.

The texture pipeline was designed to operate across the three spatial dimensions rather than relying on a single 2D image slice.

For each modality:

1. Tumour voxel intensities were taken from the normalized image.
2. Tumour intensities were quantized into **8 gray levels**.
3. The quantized tumour values were reconstructed into a 3D volume.
4. Valid neighbouring voxel pairs were identified only when both voxels belonged to the tumour mask.
5. GLCM counts were accumulated along three principal directions:
   - `(1, 0, 0)`
   - `(0, 1, 0)`
   - `(0, 0, 1)`
6. The matrix was symmetrized.
7. The GLCM was normalized so that its elements summed to 1.
8. Four texture features were then calculated.

The four features were calculated independently for all four modalities.

---

## 11. GLCM Feature Set

### FLAIR

- `FLAIR_glcm_contrast`
- `FLAIR_glcm_homogeneity`
- `FLAIR_glcm_energy`
- `FLAIR_glcm_correlation`

### T1

- `T1_glcm_contrast`
- `T1_glcm_homogeneity`
- `T1_glcm_energy`
- `T1_glcm_correlation`

### T1Gd

- `T1Gd_glcm_contrast`
- `T1Gd_glcm_homogeneity`
- `T1Gd_glcm_energy`
- `T1Gd_glcm_correlation`

### T2

- `T2_glcm_contrast`
- `T2_glcm_homogeneity`
- `T2_glcm_energy`
- `T2_glcm_correlation`

This gives:

`4 modalities × 4 texture features = 16 GLCM features`

---

## 12. GLCM Feature Definitions

### Contrast

Contrast measures the degree of local gray-level difference between neighbouring voxels.

Higher values indicate larger intensity differences between neighbouring gray levels.

### Homogeneity

Homogeneity measures how strongly the GLCM is concentrated near its diagonal.

Higher values indicate that neighbouring voxels tend to have more similar gray levels.

### Energy

Energy was calculated as:

`sqrt(sum(GLCM²))`

It reflects the concentration or uniformity of the gray-level pair distribution.

### Correlation

Correlation measures the statistical relationship between the gray levels of neighbouring voxels based on their marginal distributions.

It describes the degree to which neighbouring intensity values are linearly related.

---

# 13. Pilot Validation

Before processing all subjects, the automated pipeline was compared against the previously validated pilot workflow.

The pilot subject contained:

- **38,314 tumour voxels**
- bounding box: `58 × 59 × 48`
- centroid approximately:
  - X = `162.35`
  - Y = `110.38`
  - Z = `48.61`
- surface area approximately `14,430.06 mm²`
- sphericity approximately `0.380879`
- elongation approximately `1.150538`

The automated pipeline reproduced the expected **38-feature structure** and closely reproduced the corresponding pilot feature values.

The 3D GLCM implementation was also checked against the validated pilot values after the intensity normalization was corrected to exclude zero-valued background voxels from percentile estimation.

---

# 14. Subject-Level Output

For each successfully processed subject, a separate CSV file was generated.

The naming convention uses the original TCGA identifier:

`TCGA-02-0006_features.csv`

`TCGA-02-0009_features.csv`

`TCGA-02-0011_features.csv`

and so on.

These files are stored in:

`data/features/per_subject/`

Each individual CSV contains one row and the same **38 radiomics feature columns**.

The original pilot file:

`patient_001_features.csv`

is retained separately as the Notebook 01 pilot reference and was not used as a replacement for the TCGA subject IDs.

---

# 15. Processing Results

A total of **102 subjects** were identified in the dataset.

The automated pipeline successfully processed:

**97 subjects**

Five subjects were not processed because the required manually corrected segmentation file was unavailable.

The affected subjects were:

- `TCGA-02-0070`
- `TCGA-06-0238`
- `TCGA-08-0509`
- `TCGA-08-0520`
- `TCGA-12-3650`

These subjects contained a `GlistrBoost` segmentation but did not contain:

`GlistrBoost_ManuallyCorrected`

They were therefore not automatically substituted with the uncorrected segmentation, in order to keep the feature-extraction methodology consistent across the successfully processed cohort.

---

# 16. Excluded Subject Log

The five excluded subjects were documented separately in:

`data/features/failed_subjects.csv`

This file records:

- subject ID
- processing status
- reason for exclusion

Maintaining this record ensures that the difference between the original dataset size and the final radiomics cohort remains explicit and reproducible.

---

# 17. Master Radiomics Dataset

All successfully generated subject-level CSV files were combined into a single master table.

The master table contains:

- **97 subjects**
- **38 radiomics features**
- **1 subject identifier**

Therefore, the final table has:

**97 rows × 39 columns**

The first column is:

`subject_id`

The remaining 38 columns are the radiomics features.

The final master table is stored as:

`data/features/tcga_gbm_radiomics_features.csv`

---

# 18. Final Feature Schema

The final radiomics dataset contains the following feature groups.

### Geometry — 10 features

1. `tumour_volume_voxels`
2. `bbox_x`
3. `bbox_y`
4. `bbox_z`
5. `centroid_x`
6. `centroid_y`
7. `centroid_z`
8. `surface_area_mm2`
9. `sphericity`
10. `elongation`

### Intensity — 12 features

11. `FLAIR_mean`
12. `FLAIR_std`
13. `FLAIR_cv`
14. `T1_mean`
15. `T1_std`
16. `T1_cv`
17. `T1Gd_mean`
18. `T1Gd_std`
19. `T1Gd_cv`
20. `T2_mean`
21. `T2_std`
22. `T2_cv`

### Texture — 16 features

23. `FLAIR_glcm_contrast`
24. `FLAIR_glcm_homogeneity`
25. `FLAIR_glcm_energy`
26. `FLAIR_glcm_correlation`

27. `T1_glcm_contrast`
28. `T1_glcm_homogeneity`
29. `T1_glcm_energy`
30. `T1_glcm_correlation`

31. `T1Gd_glcm_contrast`
32. `T1Gd_glcm_homogeneity`
33. `T1Gd_glcm_energy`
34. `T1Gd_glcm_correlation`

35. `T2_glcm_contrast`
36. `T2_glcm_homogeneity`
37. `T2_glcm_energy`
38. `T2_glcm_correlation`

---

# 19. Quality Control

Several quality-control checks were performed on the final master dataset.

### Structural checks

- Number of rows: **97**
- Number of columns: **39**
- Unique subjects: **97**
- Duplicate subject IDs: **0**
- Missing values: **0**

### Numerical checks

The feature table was checked for:

- non-numeric feature values
- NaN values
- infinite values
- duplicate feature columns

All returned zero detected problems.

### Range checks

The extracted features were also checked for basic mathematical and numerical validity.

For the processed subjects:

- Sphericity was greater than 0 and not greater than 1.
- Elongation was not below 1.
- Normalized MRI means remained within the 0–1 range.
- Standard deviations were non-negative.
- Coefficients of variation were non-negative.
- GLCM contrast values were non-negative.
- GLCM homogeneity remained within 0–1.
- GLCM energy remained within 0–1.
- GLCM correlation remained within the expected -1 to 1 range.

No violations were detected in these checks.

---

# 20. Final Dataset Status

At the completion of this notebook:

**102** TCGA-GBM subjects were identified.

**97** subjects were successfully processed using the validated manually corrected tumour segmentation workflow.

**5** subjects were excluded from automated feature extraction because the manually corrected segmentation was unavailable.

Each successfully processed subject contains **38 radiomics features**.

The resulting master dataset contains:

**97 subjects × 39 columns**

with:

- 1 subject identifier
- 10 geometry features
- 12 intensity features
- 16 3D GLCM texture features

The final dataset is now organized and quality-checked for use in subsequent stages of the project.

---

# 21. Downstream Analysis

The resulting radiomics matrix provides the quantitative imaging layer for the subsequent analysis pipeline.

Potential downstream stages include:

- exploratory statistical analysis of radiomics features
- feature distributions and visualization
- feature correlation analysis
- identification of redundant features
- dimensionality reduction
- feature selection
- machine-learning modelling
- integration with clinical variables
- integration with TCGA molecular data
- radiogenomic analysis
- biological interpretation and pathway-level analysis

These downstream analyses are separate from the feature-extraction procedure performed in this notebook.

---

## Final Output Files

The completed notebook produces the following main outputs:

`data/features/per_subject/`

→ individual feature CSV for each successfully processed TCGA subject

`data/features/tcga_gbm_radiomics_features.csv`

→ combined 97-subject master radiomics dataset

`data/features/failed_subjects.csv`

→ record of the five subjects excluded from the automated pipeline

The pilot reference from Notebook 01 remains:

`data/features/patient_001_features.csv`

This notebook completes the automated generation and quality control of the initial TCGA-GBM multi-modal MRI radiomics feature dataset.